In [1]:
# Cell 1 — setup
import os
import sys
from pathlib import Path
from pprint import pprint

# Adatta questo path alla root del progetto
PROJECT_ROOT = Path("/Users/mleone1/Desktop/MoultGPT/llm").resolve()

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

os.chdir(PROJECT_ROOT)

print("PROJECT_ROOT =", PROJECT_ROOT)

PROJECT_ROOT = /Users/mleone1/Desktop/MoultGPT/llm


In [4]:
# Cell 2 — imports + one-time loading
from domain.taxonomy_graph import (
    load_taxonomy,
    build_taxonomy_graph,
    build_name_index,
    find_arthropod_taxa_in_text,
)

from pipeline.gating import (
    analyze_paper_for_moulting,
    route_query_for_paper,
)

# carica una volta sola
TAX_DF = load_taxonomy()
TAX_GRAPH = build_taxonomy_graph()
NAME_INDEX = build_name_index()

print("Taxonomy rows:", len(TAX_DF))
print("Graph nodes:", TAX_GRAPH.number_of_nodes())
print("Indexed names:", len(NAME_INDEX))

Taxonomy rows: 2396938
Graph nodes: 2396938
Indexed names: 2275139


In [74]:
from pathlib import Path
from rdflib import Graph, RDF, RDFS, OWL, URIRef, Literal

OWL_PATH = Path("./data/moultdb_moulting_ontology_v3_2.owl").resolve()

OWL_GRAPH = Graph()
OWL_GRAPH.parse(str(OWL_PATH))

print("OWL triples:", len(OWL_GRAPH))
print("OWL loaded from:", OWL_PATH)

OWL triples: 2091
OWL loaded from: /Users/mleone1/Desktop/MoultGPT/llm/data/moultdb_moulting_ontology_v3_2.owl


In [94]:
from __future__ import annotations

import re
from typing import Dict, List, Set, Tuple, Any

from rdflib import Graph, URIRef, Literal, Namespace
from rdflib.namespace import RDF, RDFS, OWL, SKOS


# =========================================================
# NAMESPACE DELL'ONTOLOGIA Moultdb
# =========================================================

MOULT = Namespace("http://purl.org/moultdb/ontology/")


# =========================================================
# CONFIG
# =========================================================

STOPWORDS_LIGHT = {
    "a", "an", "the",
    "its", "their", "his", "her",
    "this", "that", "these", "those",
    "of", "during", "in", "on", "at", "to", "for",
    "is", "are", "was", "were", "be", "been",
    "what", "which", "who", "whom", "whose",
    "how", "when", "where", "why",
    "do", "does", "did",
    "can", "could", "would", "should",
    "there", "about",
}

NOISE_CANONICALS = {
    "yes", "no", "none"
}

WEIGHT_SCORES = {
    "strong": 3.0,
    "medium": 2.0,
    "weak": 1.0,
    None: 1.0,
}

ROLE_BONUS = {
    "process": 1.0,
    "event": 1.0,
    "phase": 0.8,
    "evidence": 0.8,
    "trait": 0.4,
    "value": 0.2,
    "anatomy": 0.0,
    None: 0.0,
}


# =========================================================
# BASIC HELPERS
# =========================================================

def uri_fragment(uri: URIRef | str) -> str:
    s = str(uri)
    if "#" in s:
        return s.split("#")[-1]
    return s.rsplit("/", 1)[-1]


def norm_label(text: str) -> str:
    text = str(text).strip().lower()
    text = re.sub(r"([a-z])([A-Z])", r"\1 \2", text)
    text = text.replace("_", " ").replace("-", " ")
    text = re.sub(r"\s+", " ", text).strip()
    return text


def normalize_surface(text: str) -> str:
    """
    Normalizzazione lessicale leggera ma utile per query e surface forms.
    """
    text = str(text).strip().lower()

    # separa camelCase
    text = re.sub(r"([a-z])([A-Z])", r"\1 \2", text)

    # normalizza separatori
    text = text.replace("_", " ").replace("-", " ")

    # rimuove punteggiatura
    text = re.sub(r"[^\w\s]", " ", text)

    # compatta spazi
    text = re.sub(r"\s+", " ", text).strip()

    # normalizzazioni dominio-specifiche
    replacements = {
        "molting": "moulting",
        "molt": "moult",
        "premoult": "pre moult",
        "pre moult": "pre moult",
        "postmoult": "post moult",
        "post moult": "post moult",
        "intermolt": "intermoult",
        "instars": "instar",
        "exuviae": "exuvia",
    }

    for src, tgt in replacements.items():
        text = re.sub(rf"\b{re.escape(src)}\b", tgt, text)

    text = re.sub(r"\s+", " ", text).strip()
    return text


def generate_query_variants(text: str) -> Set[str]:
    """
    Genera più varianti della query per rendere il matching meno fragile.
    """
    base = normalize_surface(text)
    variants = {base}

    toks = base.split()

    compact = " ".join(t for t in toks if t not in STOPWORDS_LIGHT)
    compact = re.sub(r"\s+", " ", compact).strip()
    if compact:
        variants.add(compact)

    return variants


def literal_values(graph: Graph, node: URIRef, predicate: URIRef) -> List[str]:
    vals = []
    for _, _, obj in graph.triples((node, predicate, None)):
        if isinstance(obj, Literal):
            txt = str(obj).strip()
            if txt:
                vals.append(txt)
    return vals


def get_first_literal(graph: Graph, node: URIRef, predicate: URIRef) -> str | None:
    vals = literal_values(graph, node, predicate)
    return vals[0] if vals else None


def bool_from_literal(text: str | None, default: bool = False) -> bool:
    if text is None:
        return default
    return str(text).strip().lower() in {"true", "1", "yes"}


def canonical_surface_choice(surfaces: Set[str]) -> str:
    """
    Sceglie una forma canonica sensata:
    - prima multiword se più informativa
    - poi più corta
    """
    if not surfaces:
        return ""
    ordered = sorted(
        surfaces,
        key=lambda x: (
            0 if len(x.split()) > 1 else 1,
            len(x),
            x,
        )
    )
    return ordered[0]


# =========================================================
# OWL SURFACE EXTRACTION
# =========================================================

def get_all_surface_forms(graph: Graph, node: URIRef) -> Set[str]:
    """
    Raccoglie tutte le forme lessicali utili:
    - rdfs:label
    - skos:altLabel
    - skos:prefLabel
    - mdo:hasLexicalCue
    - fragment URI come fallback
    """
    surfaces: Set[str] = set()

    for v in literal_values(graph, node, RDFS.label):
        surfaces.add(v)

    for v in literal_values(graph, node, SKOS.altLabel):
        surfaces.add(v)

    for v in literal_values(graph, node, SKOS.prefLabel):
        surfaces.add(v)

    for v in literal_values(graph, node, MOULT.hasLexicalCue):
        surfaces.add(v)

    frag = uri_fragment(node)
    if frag:
        surfaces.add(frag)

    normed: Set[str] = set()
    for s in surfaces:
        s2 = normalize_surface(s)
        if s2:
            normed.add(s2)

    return normed


def find_class_by_fragment(graph: Graph, fragment_name: str) -> URIRef | None:
    target = normalize_surface(fragment_name)

    # match diretto su URI fragment
    for subj in graph.subjects(RDF.type, OWL.Class):
        frag = normalize_surface(uri_fragment(subj))
        if frag == target:
            return subj

    # fallback: match su tutte le superfici
    for subj in graph.subjects(RDF.type, OWL.Class):
        surfaces = get_all_surface_forms(graph, subj)
        if target in surfaces:
            return subj

    return None


def get_subclasses(graph: Graph, parent: URIRef) -> Set[URIRef]:
    return set(subj for subj in graph.subjects(RDFS.subClassOf, parent))


def get_all_descendants(graph: Graph, parent: URIRef) -> Set[URIRef]:
    visited: Set[URIRef] = set()
    stack = [parent]

    while stack:
        current = stack.pop()
        if current in visited:
            continue
        visited.add(current)

        children = get_subclasses(graph, current)
        for ch in children:
            if ch not in visited:
                stack.append(ch)

    return visited


def get_instances_of_classes(graph: Graph, class_nodes: Set[URIRef]) -> Set[URIRef]:
    instances: Set[URIRef] = set()
    for cls in class_nodes:
        for inst in graph.subjects(RDF.type, cls):
            instances.add(inst)
    return instances


# =========================================================
# CONCEPT FILTERING / PATCHING
# =========================================================

def should_exclude_concept(meta: Dict[str, Any]) -> bool:
    canonical = meta["canonical"]
    role = meta["role"]
    requires_context = meta["requires_context"]
    kind = meta["kind"]
    weight = meta["weight"]

    if canonical in NOISE_CANONICALS:
        return True

    # esclude individuals troppo deboli / non annotati
    if kind == "individual" and role is None and weight is None:
        return True

    # esclude concetti anatomici troppo generici che richiedono contesto
    if requires_context and role == "anatomy":
        return True

    return False


def enrich_surfaces_with_manual_bridges(meta: Dict[str, Any]) -> None:
    """
    Patch leggera per coprire alcune formulazioni realistiche.
    Non sostituisce l'ontologia, ma evita buchi fragili.
    """
    surfaces = set(meta["surfaces"])
    canonical = meta["canonical"]

    # ponte per instar
    if canonical in {"instar", "instar transition"} or "instar" in surfaces:
        surfaces.update({
            "multiple instar",
            "developmental stage between moult",
        })

    # ponte per ecdysis / moult
    if {"ecdysis", "moult", "moulting"} & surfaces:
        surfaces.update({
            "shedding exoskeleton",
            "shed exoskeleton",
        })

    meta["surfaces"] = sorted(normalize_surface(s) for s in surfaces if s)


# =========================================================
# BUILD ONTOLOGY INDEX FOR GATING
# =========================================================

def build_moulting_ontology_terms(graph: Graph, root_class_names: List[str]):
    """
    Restituisce:
    - surface_index: dict surface -> list of concept metas
    - concepts: dict uri -> meta
    - selected_classes
    - selected_instances
    """
    selected_classes: Set[URIRef] = set()

    for name in root_class_names:
        cls = find_class_by_fragment(graph, name)
        if cls is None:
            print(f"[WARN] root class not found: {name}")
            continue
        selected_classes |= get_all_descendants(graph, cls)

    selected_instances = get_instances_of_classes(graph, selected_classes)

    concepts: Dict[str, Dict[str, Any]] = {}

    def register_node(node: URIRef, kind: str) -> None:
        uri = str(node)
        surfaces = get_all_surface_forms(graph, node)
        if not surfaces:
            return

        role = get_first_literal(graph, node, MOULT.hasSemanticRole)
        weight = get_first_literal(graph, node, MOULT.hasGatingWeight)
        requires_context_raw = get_first_literal(graph, node, MOULT.requiresContext)

        meta = {
            "uri": uri,
            "kind": kind,
            "surfaces": sorted(surfaces),
            "role": normalize_surface(role) if role else None,
            "weight": normalize_surface(weight) if weight else None,
            "requires_context": bool_from_literal(requires_context_raw, default=False),
            "canonical": canonical_surface_choice(surfaces),
        }

        enrich_surfaces_with_manual_bridges(meta)

        if should_exclude_concept(meta):
            return

        concepts[uri] = meta

    for node in selected_classes:
        register_node(node, "class")

    for node in selected_instances:
        register_node(node, "individual")

    surface_index: Dict[str, List[Dict[str, Any]]] = {}
    for meta in concepts.values():
        for surf in meta["surfaces"]:
            surface_index.setdefault(surf, []).append(meta)

    return surface_index, concepts, selected_classes, selected_instances


# =========================================================
# MATCHING
# =========================================================

def build_ngrams(tokens: List[str], max_n: int = 5) -> Set[str]:
    grams: Set[str] = set()
    for n in range(1, max_n + 1):
        if len(tokens) >= n:
            grams |= {" ".join(tokens[i:i+n]) for i in range(len(tokens) - n + 1)}
    return grams


def concept_match_score(meta: Dict[str, Any], matched_surface: str) -> float:
    """
    Score del singolo concetto matchato.
    """
    weight_score = WEIGHT_SCORES.get(meta["weight"], 1.0)
    role_bonus = ROLE_BONUS.get(meta["role"], 0.0)

    # piccola preferenza per match multiword
    lexical_bonus = 0.3 if len(matched_surface.split()) > 1 else 0.0

    # penalizza concetti che richiedono contesto
    context_penalty = -0.8 if meta["requires_context"] else 0.0

    return weight_score + role_bonus + lexical_bonus + context_penalty


def match_ontology_terms(text: str, ontology_surface_index: Dict[str, List[Dict[str, Any]]]):
    """
    Matcha superfici ontologiche usando:
    - query normalizzata
    - query alleggerita dalle stopwords leggere
    - ngrammi fino a 5 token

    Deduplica per (matched_surface, canonical) invece che per URI
    per evitare triplicati quasi identici.
    """
    hits = []
    seen: Set[Tuple[str, str]] = set()

    for text_n in generate_query_variants(text):
        tokens = text_n.split()
        grams = build_ngrams(tokens, max_n=5)

        for gram in grams:
            if gram not in ontology_surface_index:
                continue

            for meta in ontology_surface_index[gram]:
                key = (gram, meta["canonical"])
                if key in seen:
                    continue

                hit = {
                    "matched_surface": gram,
                    "canonical": meta["canonical"],
                    "uri": meta["uri"],
                    "kind": meta["kind"],
                    "role": meta["role"],
                    "weight": meta["weight"],
                    "requires_context": meta["requires_context"],
                    "score": concept_match_score(meta, gram),
                }
                hits.append(hit)
                seen.add(key)

    hits = sorted(
        hits,
        key=lambda h: (
            -h["score"],
            -len(h["matched_surface"].split()),
            h["matched_surface"],
        )
    )
    return hits


def summarize_hits(hits: List[Dict[str, Any]]) -> List[Dict[str, Any]]:
    return [
        {
            "matched_surface": h["matched_surface"],
            "canonical": h["canonical"],
            "role": h["role"],
            "weight": h["weight"],
            "requires_context": h["requires_context"],
            "score": round(h["score"], 2),
        }
        for h in hits
    ]


# =========================================================
# QUERY DECISION
# =========================================================

def query_has_moulting_signal_from_ontology(
    query: str,
    ontology_surface_index: Dict[str, List[Dict[str, Any]]],
    min_hits: int = 1,
    min_score: float = 2.5,
):
    """
    Decisione più robusta del semplice n_hits >= 1.

    allow = True se:
    - almeno min_hits match
    - e score totale >= min_score

    In questo modo eviti che un singolo termine debole/contestuale
    apra il gate da solo.
    """
    hits = match_ontology_terms(query, ontology_surface_index)
    total_score = sum(h["score"] for h in hits)

    allow = (len(hits) >= min_hits) and (total_score >= min_score)

    return {
        "allow": allow,
        "n_hits": len(hits),
        "score": round(total_score, 2),
        "hits": hits,
    }


# =========================================================
# ROOT CLASSES CONSIGLIATE PER IL QUERY GATING
# =========================================================

ROOT_ONTO_CLASSES = [
    "MoultingProcess",
    "MoultingEvent",
    "MoultingPhase",
    "Trait",
    "TraitValue",
    "Context",
    "Instar",
    "Exuvia",
]

# Nota:
# volutamente NON includo "AnatomicalStructure" nel query gate,
# perché introduce falsi positivi da termini generici come
# "exoskeleton" o "cuticle" presi da soli.


# =========================================================
# OPTIONAL DEBUG HELPERS
# =========================================================

def debug_print_namespaces(graph: Graph) -> None:
    print("\nNAMESPACES IN GRAPH")
    for prefix, ns in graph.namespaces():
        print(prefix, "->", ns)


def debug_probe_terms(ontology_surface_index: Dict[str, List[Dict[str, Any]]], probe_terms: List[str]) -> None:
    print("\nCHECK SURFACES IN INDEX")
    for term in probe_terms:
        t = normalize_surface(term)
        print(f"{term!r} -> normalized: {t!r} -> present: {t in ontology_surface_index}")
        if t in ontology_surface_index:
            for meta in ontology_surface_index[t]:
                print(
                    {
                        "canonical": meta["canonical"],
                        "kind": meta["kind"],
                        "role": meta["role"],
                        "weight": meta["weight"],
                        "requires_context": meta["requires_context"],
                        "uri": meta["uri"],
                    }
                )
            print("-" * 60)

In [95]:
ONTO_TERMS, ONTO_CONCEPTS, ONTO_CLASSES, ONTO_INSTANCES = build_moulting_ontology_terms(
    OWL_GRAPH,
    ROOT_ONTO_CLASSES
)

print("Selected ontology classes:", len(ONTO_CLASSES))
print("Selected ontology instances:", len(ONTO_INSTANCES))
print("Ontology surface forms for gating:", len(ONTO_TERMS))
print("Ontology concepts indexed:", len(ONTO_CONCEPTS))

Selected ontology classes: 90
Selected ontology instances: 171
Ontology surface forms for gating: 194
Ontology concepts indexed: 90


In [103]:
from collections import Counter
from typing import List, Tuple, Dict, Any


def evaluate_query_gating(
    labeled_queries: List[Tuple[str, bool]],
    onto_terms,
    min_hits: int = 1,
    min_score: float = 2.5,
    verbose: bool = True,
) -> Dict[str, Any]:
    """
    Evaluate ontology-based moulting query gating.

    Parameters
    ----------
    labeled_queries : list of (query, gold_label)
        gold_label=True  -> query should be allowed
        gold_label=False -> query should be blocked
    onto_terms : any
        Ontology terms object used by query_has_moulting_signal_from_ontology
    min_hits : int
        Minimum number of hits required
    min_score : float
        Minimum accumulated score required
    verbose : bool
        If True, print detailed report

    Returns
    -------
    dict with metrics and detailed predictions
    """

    results = []

    for query, gold in labeled_queries:
        pred = query_has_moulting_signal_from_ontology(
            query,
            onto_terms,
            min_hits=min_hits,
            min_score=min_score,
        )

        allow = bool(pred["allow"])
        score = float(pred.get("score", 0.0))
        n_hits = int(pred.get("n_hits", 0))
        hits = pred.get("hits", [])

        results.append({
            "query": query,
            "gold": gold,
            "pred": allow,
            "correct": allow == gold,
            "score": score,
            "n_hits": n_hits,
            "hits": hits,
        })

    # confusion matrix
    tp = sum(1 for r in results if r["gold"] is True and r["pred"] is True)
    tn = sum(1 for r in results if r["gold"] is False and r["pred"] is False)
    fp = sum(1 for r in results if r["gold"] is False and r["pred"] is True)
    fn = sum(1 for r in results if r["gold"] is True and r["pred"] is False)

    total = len(results)
    accuracy = (tp + tn) / total if total else 0.0
    precision = tp / (tp + fp) if (tp + fp) else 0.0
    recall = tp / (tp + fn) if (tp + fn) else 0.0
    f1 = (2 * precision * recall / (precision + recall)) if (precision + recall) else 0.0
    specificity = tn / (tn + fp) if (tn + fp) else 0.0

    false_positives = [r for r in results if r["gold"] is False and r["pred"] is True]
    false_negatives = [r for r in results if r["gold"] is True and r["pred"] is False]

    summary = {
        "n_total": total,
        "n_positive_gold": sum(1 for _, g in labeled_queries if g is True),
        "n_negative_gold": sum(1 for _, g in labeled_queries if g is False),
        "tp": tp,
        "tn": tn,
        "fp": fp,
        "fn": fn,
        "accuracy": accuracy,
        "precision": precision,
        "recall": recall,
        "f1": f1,
        "specificity": specificity,
        "false_positives": false_positives,
        "false_negatives": false_negatives,
        "results": results,
    }

    if verbose:
        print("\n" + "=" * 100)
        print("GATING EVALUATION REPORT")
        print("=" * 100)
        print(f"Total queries     : {total}")
        print(f"Gold positives    : {summary['n_positive_gold']}")
        print(f"Gold negatives    : {summary['n_negative_gold']}")
        print("-" * 100)
        print(f"TP: {tp} | TN: {tn} | FP: {fp} | FN: {fn}")
        print("-" * 100)
        print(f"Accuracy          : {accuracy:.4f}")
        print(f"Precision         : {precision:.4f}")
        print(f"Recall            : {recall:.4f}")
        print(f"F1                : {f1:.4f}")
        print(f"Specificity       : {specificity:.4f}")

        print("\n" + "=" * 100)
        print("FALSE POSITIVES (should have been blocked, but were allowed)")
        print("=" * 100)
        if false_positives:
            for i, r in enumerate(false_positives, 1):
                print(f"\n[FP {i}] {r['query']}")
                print(f"score={r['score']:.2f} | n_hits={r['n_hits']}")
                print("hits:", summarize_hits(r["hits"]))
        else:
            print("None.")

        print("\n" + "=" * 100)
        print("FALSE NEGATIVES (should have been allowed, but were blocked)")
        print("=" * 100)
        if false_negatives:
            for i, r in enumerate(false_negatives, 1):
                print(f"\n[FN {i}] {r['query']}")
                print(f"score={r['score']:.2f} | n_hits={r['n_hits']}")
                print("hits:", summarize_hits(r["hits"]))
        else:
            print("None.")

        print("\n" + "=" * 100)
        print("ALL PREDICTIONS")
        print("=" * 100)
        for i, r in enumerate(results, 1):
            status = "OK" if r["correct"] else "ERR"
            print(
                f"[{i:02d}] {status} | gold={r['gold']} pred={r['pred']} "
                f"| score={r['score']:.2f} | hits={r['n_hits']} | {r['query']}"
            )

    return summary

In [110]:
test_queries_extended = [
    # ✅ MOULTING DIRETTO
    ("Describe the moulting behaviour observed in this species.", True),
    ("What is the sequence of events during ecdysis?", True),
    ("Are there signs of recent moulting in the specimen?", True),
    ("How is the old cuticle removed during moulting?", True),
    ("Is there any indication of moulting-related mortality?", True),
    ("Do individuals leave behind exuviae after moulting?", True),
    ("What structures are involved in the moulting process?", True),
    ("Describe the moulting stages identified in the study.", True),

    # ✅ MOULTING INDIRETTO / SUBTLE
    ("Does the organism undergo periodic cuticle renewal?", True),
    ("Are there developmental stages involving shedding?", True),
    ("Is there evidence of repeated growth phases separated by structural changes?", True),
    ("Do specimens show discontinuous growth patterns?", True),
    ("Is body enlargement associated with cuticular loss?", True),
    ("Are there preserved remains of previous body coverings?", True),

    # ⚠️ BORDERLINE POSITIVES
    ("Describe ontogenetic changes in this species.", True),
    ("How does the organism increase in size over time?", True),
    ("Are there discrete growth stages in the lifecycle?", True),
    ("Does development occur in stages or continuously?", True),

    # ❌ ARTHROPODS MA NON MOULTING
    ("What feeding strategies are observed in these arthropods?", False),
    ("Describe the appendages of the organism.", False),
    ("What is the taxonomic classification of the species?", False),
    ("How do these organisms reproduce?", False),
    ("What environmental conditions do these arthropods prefer?", False),
    ("What predators interact with this species?", False),

    # ❌ SCIENTIFIC BUT OFF-TOPIC
    ("What is the chemical composition of the sediment?", False),
    ("Describe the geological formation where fossils were found.", False),
    ("What dating method was used for the samples?", False),
    ("Explain the statistical model used in the analysis.", False),

    # ❌ TOTALMENTE FUORI DOMINIO
    ("What is the capital of Germany?", False),
    ("Explain general relativity.", False),
    ("How does a neural network learn?", False),
    ("What is blockchain technology?", False),

    # ⚠️ VERTEBRATI: per ora li consideriamo TRUE
    ("How do reptiles perform moulting?", True),
    ("Do birds experience moulting cycles annually?", True),
    ("What hormonal control regulates moulting in mammals?", True),

    # ❌ AMBIGUE / TROPPO GENERICHE
    ("What transformations occur in this organism?", False),
    ("Does the organism change form over time?", False),
    ("Describe the lifecycle of the species.", False),
    ("What stages are present in its life?", False),

    # ❌ ADVERSARIAL / NON BIOLOGICAL
    ("Explain moulting in a metaphorical sense.", False),
    ("Is moulting used as an analogy in this paper?", False),
    ("Does the text mention moulting in a non-biological context?", False),
    ("What is data moulting in machine learning?", False),

    # ❌ HARD NEGATIVES
    ("Describe how the organism regenerates limbs.", False),
    ("Is there evidence of metamorphosis?", False),
    ("How does the organism reproduce sexually?", False),
    ("What is the lifespan of the species?", False),
]

In [111]:
task_queries = [
    ("Extract all moulting-related traits from this paper.", True),
    ("List all moulting features reported for the species.", True),
    ("Return all labels associated with the moulting process.", True),
    ("Extract the moulting stage described in the article.", True),
    ("Extract the location of the moulting suture.", True),
    ("Summarise all evidence related to exuviation.", True),
    ("Give me every trait in the paper that concerns moulting.", True),

    ("Extract the habitat of the species.", False),
    ("Summarise the stratigraphic context.", False),
    ("What statistical tests were used?", False),
    ("Explain moulting as a metaphor.", False),
]

In [113]:
report = evaluate_query_gating(
    labeled_queries=test_queries_extended,
    onto_terms=ONTO_TERMS,
    min_hits=1,
    min_score=2.5,
    verbose=True,
)


GATING EVALUATION REPORT
Total queries     : 47
Gold positives    : 21
Gold negatives    : 26
----------------------------------------------------------------------------------------------------
TP: 13 | TN: 22 | FP: 4 | FN: 8
----------------------------------------------------------------------------------------------------
Accuracy          : 0.7447
Precision         : 0.7647
Recall            : 0.6190
F1                : 0.6842
Specificity       : 0.8462

FALSE POSITIVES (should have been blocked, but were allowed)

[FP 1] Explain moulting in a metaphorical sense.
score=8.00 | n_hits=2
hits: [{'matched_surface': 'moulting', 'canonical': 'ecdysial event', 'role': 'process', 'weight': 'strong', 'requires_context': False, 'score': 4.0}, {'matched_surface': 'moulting', 'canonical': 'moulting process', 'role': 'process', 'weight': 'strong', 'requires_context': False, 'score': 4.0}]

[FP 2] Is moulting used as an analogy in this paper?
score=8.00 | n_hits=2
hits: [{'matched_surface': 'm

In [116]:
from domain.taxonomic_scope import (
    load_taxonomy_records,
    build_taxonomy_name_index,
    analyze_taxonomic_scope,
    build_paper_profile,
)

records = load_taxonomy_records("data/arthropod_taxonomy.csv")
name_index = build_taxonomy_name_index(records)

print("taxonomy records:", len(records))
print("name index size:", len(name_index))

taxonomy records: 2396938
name index size: 4129838


In [117]:
text = """
Trilobite exuviae record the development of individual trilobites and their molting process.
The study focuses on Omegops sp. A, a phacopid trilobite from the Hongguleleng Formation.
"""

tax_profile = analyze_taxonomic_scope(text, name_index)
print(tax_profile)

{'detected_taxa': ['Omegops'], 'matches': [{'canonical_name': 'Omegops', 'path': '1.172.50.16.16', 'depth': 5, 'matched_surfaces': ['omegops'], 'match_count': 1}], 'arthropod_score': 0.28, 'non_arthropod_score': 0.0, 'negative_hits': [], 'taxonomic_signal_state': 'arthropod_weakly_supported', 'score_breakdown': {'specificity': 0.25, 'coherence': 0.0, 'repetition': 0.03}}


In [13]:
import os
import sys
import importlib

PROJECT_ROOT = "/Users/mleone1/Desktop/MoultGPT/llm"
sys.path.append(PROJECT_ROOT)

import domain.optimized_taxonomy_lookup as otl
importlib.reload(otl)

lookup = otl.TaxonomyLookup(
    csv_path=os.path.join(PROJECT_ROOT, "data", "arthropod_taxonomy.csv"),
    pickle_path=os.path.join(PROJECT_ROOT, "data", "taxonomy_lookup.pkl"),
    rebuild=True   # solo la prima volta
)

[TaxonomyLookup] Building lookup from CSV (slow, one-time)...
[TaxonomyLookup] Rows read: 2396938
[TaxonomyLookup] Unique normalized names: 2444609
[TaxonomyLookup] Paths indexed: 2396938
[TaxonomyLookup] Compiled regex patterns: 2444609
[TaxonomyLookup] Saved compiled lookup to: /Users/mleone1/Desktop/MoultGPT/llm/data/taxonomy_lookup.pkl


In [14]:
sample = """
trilobita exuviae record the development of individual trilobites and their molting process.
The study focuses on Omegops sp. A, a phacopid arthropoda, spiders trilobite from the Hongguleleng Formation.
"""

print(lookup.has_any_match(sample))

print("\nDIRECT")
for m in lookup.find_taxa_in_text(sample):
    print(m)

print("\nWITH ANCESTORS")
for m in lookup.find_taxa_with_ancestors(sample):
    print(m)

True

DIRECT
{'matched_name': 'omegops', 'canonical_name': 'omegops', 'taxon_id': 3596, 'path': '1.172.50.16.16', 'depth': 5}
{'matched_name': 'spiders', 'canonical_name': 'araneae', 'taxon_id': 808, 'path': '1.35.1.11', 'depth': 4}
{'matched_name': 'trilobita', 'canonical_name': 'trilobita', 'taxon_id': 173, 'path': '1.172', 'depth': 2}
{'matched_name': 'arthropoda', 'canonical_name': 'arthropoda', 'taxon_id': 1, 'path': '1', 'depth': 1}

WITH ANCESTORS
{'matched_name': 'omegops', 'canonical_name': 'omegops', 'taxon_id': 3596, 'path': '1.172.50.16.16', 'depth': 5}
{'matched_name': 'phacopidae', 'canonical_name': 'phacopidae', 'taxon_id': 1326, 'path': '1.172.50.16', 'depth': 4}
{'matched_name': 'araneae', 'canonical_name': 'araneae', 'taxon_id': 808, 'path': '1.35.1.11', 'depth': 4}
{'matched_name': 'phacopida', 'canonical_name': 'phacopida', 'taxon_id': 479, 'path': '1.172.50', 'depth': 3}
{'matched_name': 'arachnida', 'canonical_name': 'arachnida', 'taxon_id': 239, 'path': '1.35.1',

In [17]:
import os
import sys
import importlib

PROJECT_ROOT = "/Users/mleone1/Desktop/MoultGPT/llm"
sys.path.append(PROJECT_ROOT)

import domain.optimized_taxonomy_lookup as taxmod
import domain.moulting_ontology_gate as moultmod
import domain.domain_gate as gate_mod

importlib.reload(taxmod)
importlib.reload(moultmod)
importlib.reload(gate_mod)

<module 'domain.domain_gate' from '/Users/mleone1/Desktop/MoultGPT/llm/domain/domain_gate.py'>

In [18]:
taxonomy_lookup = taxmod.TaxonomyLookup(
    csv_path=os.path.join(PROJECT_ROOT, "data", "arthropod_taxonomy.csv"),
    pickle_path=os.path.join(PROJECT_ROOT, "data", "taxonomy_lookup.pkl"),
    rebuild=False,
)

ontology_gate = moultmod.MoultingOntologyGate(
    owl_path=os.path.join(PROJECT_ROOT, "data", "moultdb_moulting_ontology_v3_2.owl")
)

[TaxonomyLookup] Loading precomputed lookup...
[TaxonomyLookup] Names loaded: 2444609
[TaxonomyLookup] Paths loaded: 2396938
[TaxonomyLookup] Patterns loaded: 2444609


In [19]:
paper_text = """
Trilobite exuviae record the development of individual trilobites and their molting process.
The study focuses on Omegops sp. A, a phacopid trilobite from the Hongguleleng Formation.
"""

user_query = "Extract all moulting-related traits of this species."

In [20]:
result = gate_mod.analyze_paper_and_query_domain(
    paper_text=paper_text,
    user_query=user_query,
    taxonomy_lookup=taxonomy_lookup,
    ontology_gate=ontology_gate,
    min_query_hits=1,
    min_query_score=2.5,
)

result

{'allow': True,
 'final_label': 'in_scope',
 'message': 'Paper contains arthropod signal and query concerns moulting.',
 'paper_gate': {'allow': True,
  'label': 'arthropod_detected',
  'n_direct_matches': 1,
  'n_propagated_matches': 5,
  'direct_matches': [{'matched_name': 'omegops',
    'canonical_name': 'omegops',
    'taxon_id': 3596,
    'path': '1.172.50.16.16',
    'depth': 5}],
  'propagated_matches': [{'matched_name': 'omegops',
    'canonical_name': 'omegops',
    'taxon_id': 3596,
    'path': '1.172.50.16.16',
    'depth': 5},
   {'matched_name': 'phacopidae',
    'canonical_name': 'phacopidae',
    'taxon_id': 1326,
    'path': '1.172.50.16',
    'depth': 4},
   {'matched_name': 'phacopida',
    'canonical_name': 'phacopida',
    'taxon_id': 479,
    'path': '1.172.50',
    'depth': 3},
   {'matched_name': 'trilobita',
    'canonical_name': 'trilobita',
    'taxon_id': 173,
    'path': '1.172',
    'depth': 2},
   {'matched_name': 'arthropoda',
    'canonical_name': 'arthr

In [21]:
print("ALLOW:", result["allow"])
print("FINAL LABEL:", result["final_label"])
print("MESSAGE:", result["message"])

print("\n=== PAPER GATE ===")
print("allow:", result["paper_gate"]["allow"])
print("label:", result["paper_gate"]["label"])
print("n_direct_matches:", result["paper_gate"]["n_direct_matches"])
print("n_propagated_matches:", result["paper_gate"]["n_propagated_matches"])
for x in result["paper_gate"]["propagated_matches"][:15]:
    print(x)

print("\n=== QUERY GATE ===")
print("allow:", result["query_gate"]["allow"])
print("label:", result["query_gate"]["label"])
print("n_hits:", result["query_gate"]["n_hits"])
print("score:", result["query_gate"]["score"])
for h in result["query_gate"]["summary_hits"][:20]:
    print(h)

ALLOW: True
FINAL LABEL: in_scope
MESSAGE: Paper contains arthropod signal and query concerns moulting.

=== PAPER GATE ===
allow: True
label: arthropod_detected
n_direct_matches: 1
n_propagated_matches: 5
{'matched_name': 'omegops', 'canonical_name': 'omegops', 'taxon_id': 3596, 'path': '1.172.50.16.16', 'depth': 5}
{'matched_name': 'phacopidae', 'canonical_name': 'phacopidae', 'taxon_id': 1326, 'path': '1.172.50.16', 'depth': 4}
{'matched_name': 'phacopida', 'canonical_name': 'phacopida', 'taxon_id': 479, 'path': '1.172.50', 'depth': 3}
{'matched_name': 'trilobita', 'canonical_name': 'trilobita', 'taxon_id': 173, 'path': '1.172', 'depth': 2}
{'matched_name': 'arthropoda', 'canonical_name': 'arthropoda', 'taxon_id': 1, 'path': '1', 'depth': 1}

=== QUERY GATE ===
allow: True
label: moulting_query_detected
n_hits: 2
score: 8.0
{'matched_surface': 'moulting', 'canonical': 'moulting process', 'role': 'process', 'weight': 'strong', 'requires_context': False, 'score': 4.0}
{'matched_surfac

In [22]:
test_cases = [
    {
        "id": 1,
        "paper_text": "The study focuses on Omegops sp. A, a phacopid trilobite. Exuviae and moulting are discussed.",
        "user_query": "Extract all moulting-related traits of this species.",
        "expected": True,
        "note": "arthropod + explicit moulting"
    },
    {
        "id": 2,
        "paper_text": "Trilobite exuviae record the development of individual trilobites and their molting process.",
        "user_query": "What evidence of ecdysis is reported in this paper?",
        "expected": True,
        "note": "arthropod + ecdysis"
    },
    {
        "id": 3,
        "paper_text": "Omegops sp. A is described from the Hongguleleng Formation.",
        "user_query": "Describe the moulting process in this species.",
        "expected": True,
        "note": "arthropod from taxonomy only + moulting query"
    },
    {
        "id": 4,
        "paper_text": "The specimen belongs to Phacopidae and preserves exuviae.",
        "user_query": "Does the paper describe exuviae?",
        "expected": True,
        "note": "arthropod family + exuvia query"
    },
    {
        "id": 5,
        "paper_text": "Araneae are discussed together with moulting phases and cuticular shedding.",
        "user_query": "Summarize the moulting phases reported here.",
        "expected": True,
        "note": "spider order + moulting phases"
    },
    {
        "id": 6,
        "paper_text": "The arthropoda specimen shows repeated exuvial configurations.",
        "user_query": "Extract all information about moulting in this paper.",
        "expected": True,
        "note": "generic arthropoda + strong moulting query"
    },

    {
        "id": 7,
        "paper_text": "The study focuses on Omegops sp. A, a phacopid trilobite.",
        "user_query": "What habitat is described in this paper?",
        "expected": False,
        "note": "arthropod paper + non-moulting query"
    },
    {
        "id": 8,
        "paper_text": "The specimen belongs to Phacopidae and Trilobita.",
        "user_query": "Who collected the samples?",
        "expected": False,
        "note": "arthropod paper + metadata query"
    },
    {
        "id": 9,
        "paper_text": "Spiders were sampled across several cave systems.",
        "user_query": "Summarize the geographic distribution.",
        "expected": False,
        "note": "arthropod paper + geography query"
    },
    {
        "id": 10,
        "paper_text": "Arthropoda diversity was surveyed in the region.",
        "user_query": "What methods were used for fossil preparation?",
        "expected": False,
        "note": "arthropod signal + no moulting in query"
    },
    {
        "id": 11,
        "paper_text": "Omegops sp. A and Phacopidae are compared morphologically.",
        "user_query": "Describe the phylogenetic implications.",
        "expected": False,
        "note": "arthropod taxonomy + non-moulting scientific query"
    },
    {
        "id": 12,
        "paper_text": "The trilobite material comes from the upper member formation.",
        "user_query": "Extract all characters of this species.",
        "expected": False,
        "note": "deictic but no moulting signal in query"
    },

    {
        "id": 13,
        "paper_text": "Snakes shed their skin during growth and seasonal change.",
        "user_query": "Describe the moulting process in this species.",
        "expected": False,
        "note": "vertebrate + moulting query"
    },
    {
        "id": 14,
        "paper_text": "Birds replace feathers during seasonal molt.",
        "user_query": "What evidence of ecdysis is reported?",
        "expected": False,
        "note": "bird feather molt but no arthropod paper"
    },
    {
        "id": 15,
        "paper_text": "Lizard skin shedding was observed under dry conditions.",
        "user_query": "Extract all moulting-related traits.",
        "expected": False,
        "note": "reptile shedding + moulting query"
    },
    {
        "id": 16,
        "paper_text": "The mammal develops rapidly after birth.",
        "user_query": "Summarize exuviae and instar information.",
        "expected": False,
        "note": "non-arthropod paper + strong moulting query"
    },
    {
        "id": 17,
        "paper_text": "A fish ecology study measured seasonal abundance.",
        "user_query": "Does the paper discuss post-moult stages?",
        "expected": False,
        "note": "non-arthropod paper + moulting query"
    },
    {
        "id": 18,
        "paper_text": "The vertebrate skeleton was reconstructed from fossils.",
        "user_query": "What moulting phase is described?",
        "expected": False,
        "note": "non-arthropod fossil paper + moulting query"
    },

    {
        "id": 19,
        "paper_text": "This study examines alpine climate trends over 30 years.",
        "user_query": "What habitat is described?",
        "expected": False,
        "note": "fully out of scope"
    },
    {
        "id": 20,
        "paper_text": "Economic development in coastal regions was analyzed.",
        "user_query": "Summarize the results.",
        "expected": False,
        "note": "fully out of scope"
    },
    {
        "id": 21,
        "paper_text": "Sedimentary facies were described from the Hongguleleng Formation.",
        "user_query": "Extract all moulting traits.",
        "expected": False,
        "note": "geology paper + moulting query"
    },
    {
        "id": 22,
        "paper_text": "Growth dynamics were modeled in plant populations.",
        "user_query": "Describe instar transitions.",
        "expected": False,
        "note": "plant paper + moulting query"
    },

    {
        "id": 23,
        "paper_text": "The study describes Omegops sp. A and associated exuviae in trilobites.",
        "user_query": "Describe the developmental stage.",
        "expected": False,
        "note": "borderline query, probably too vague"
    },
    {
        "id": 24,
        "paper_text": "Crustaceans were sampled and several exuviae were preserved.",
        "user_query": "Is moulting discussed in this paper?",
        "expected": True,
        "note": "simple yes/no but in-scope"
    },
]

In [23]:
results = []

for case in test_cases:
    result = gate_mod.analyze_paper_and_query_domain(
        paper_text=case["paper_text"],
        user_query=case["user_query"],
        taxonomy_lookup=taxonomy_lookup,
        ontology_gate=ontology_gate,
        min_query_hits=1,
        min_query_score=2.5,
    )

    pred = result["allow"]
    ok = pred == case["expected"]

    results.append({
        "id": case["id"],
        "expected": case["expected"],
        "predicted": pred,
        "ok": ok,
        "final_label": result["final_label"],
        "paper_label": result["paper_gate"]["label"],
        "query_label": result["query_gate"]["label"],
        "paper_direct": result["paper_gate"]["n_direct_matches"],
        "paper_propagated": result["paper_gate"]["n_propagated_matches"],
        "query_hits": result["query_gate"]["n_hits"],
        "query_score": result["query_gate"]["score"],
        "note": case["note"],
        "paper_text": case["paper_text"],
        "user_query": case["user_query"],
    })

In [24]:
for r in results:
    status = "OK" if r["ok"] else "FAIL"
    print(
        f"[{r['id']:02d}] {status} | expected={r['expected']} pred={r['predicted']} "
        f"| label={r['final_label']} | paper={r['paper_label']} "
        f"| query={r['query_label']} | qscore={r['query_score']}"
    )

[01] OK | expected=True pred=True | label=in_scope | paper=arthropod_detected | query=moulting_query_detected | qscore=8.0
[02] FAIL | expected=True pred=False | label=paper_out_of_scope | paper=no_arthropod_signal | query=moulting_query_detected | qscore=4.0
[03] OK | expected=True pred=True | label=in_scope | paper=arthropod_detected | query=moulting_query_detected | qscore=12.3
[04] OK | expected=True pred=True | label=in_scope | paper=arthropod_detected | query=moulting_query_detected | qscore=3.8
[05] OK | expected=True pred=True | label=in_scope | paper=arthropod_detected | query=moulting_query_detected | qscore=8.0
[06] OK | expected=True pred=True | label=in_scope | paper=arthropod_detected | query=moulting_query_detected | qscore=8.0
[07] OK | expected=False pred=False | label=query_out_of_scope | paper=arthropod_detected | query=no_moulting_signal_in_query | qscore=0
[08] OK | expected=False pred=False | label=query_out_of_scope | paper=arthropod_detected | query=no_moulting_

In [25]:
errors = [r for r in results if not r["ok"]]

print(f"Total errors: {len(errors)}\n")
for r in errors:
    print("=" * 100)
    print(f"CASE {r['id']} | expected={r['expected']} predicted={r['predicted']}")
    print("NOTE:", r["note"])
    print("FINAL LABEL:", r["final_label"])
    print("PAPER LABEL:", r["paper_label"], "| direct:", r["paper_direct"], "| propagated:", r["paper_propagated"])
    print("QUERY LABEL:", r["query_label"], "| hits:", r["query_hits"], "| score:", r["query_score"])
    print("PAPER:", r["paper_text"])
    print("QUERY:", r["user_query"])

Total errors: 2

CASE 2 | expected=True predicted=False
NOTE: arthropod + ecdysis
FINAL LABEL: paper_out_of_scope
PAPER LABEL: no_arthropod_signal | direct: 0 | propagated: 0
QUERY LABEL: moulting_query_detected | hits: 1 | score: 4.0
PAPER: Trilobite exuviae record the development of individual trilobites and their molting process.
QUERY: What evidence of ecdysis is reported in this paper?
CASE 21 | expected=False predicted=True
NOTE: geology paper + moulting query
FINAL LABEL: in_scope
PAPER LABEL: arthropod_detected | direct: 1 | propagated: 21
QUERY LABEL: moulting_query_detected | hits: 2 | score: 8.0
PAPER: Sedimentary facies were described from the Hongguleleng Formation.
QUERY: Extract all moulting traits.
